# Water Quality Data Cleaning: EPA Stations

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [2]:
data_path = r'../../../../data/tabular/water-quality/raw'
df_station = pd.read_csv(f'{data_path}/epa-stations.csv')

In [3]:
# Step 1: Inspect
print(df_station.shape)
df_station.head()

(621, 37)


,OrganizationIdentifier,OrganizationFormalName,MonitoringLocationIdentifier,MonitoringLocationName,MonitoringLocationTypeName,MonitoringLocationDescriptionText,HUCEightDigitCode,DrainageAreaMeasure/MeasureValue,DrainageAreaMeasure/MeasureUnitCode,ContributingDrainageAreaMeasure/MeasureValue,...,AquiferName,LocalAqfrName,FormationTypeText,AquiferTypeName,ConstructionDateText,WellDepthMeasure/MeasureValue,WellDepthMeasure/MeasureUnitCode,WellHoleDepthMeasure/MeasureValue,WellHoleDepthMeasure/MeasureUnitCode,ProviderName
0,USGS-IA,USGS Iowa Water Science Center,USGS-05420500,"Mississippi River at Clinton, IA",Stream,NaN,7080101,85600.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
1,USGS-IA,USGS Iowa Water Science Center,USGS-05451210,"South Fork Iowa River NE of New Providence, IA",Stream,NaN,7080207,224.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
2,USGS-IA,USGS Iowa Water Science Center,USGS-05465500,"Iowa River at Wapello, IA",Stream,NaN,7080209,12500.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
3,USGS-IA,USGS Iowa Water Science Center,USGS-05474000,"Skunk River at Augusta, IA",Stream,NaN,7080107,4312.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS
4,USGS-IA,USGS Iowa Water Science Center,USGS-05490500,"Des Moines River at Keosauqua, IA",Stream,NaN,7100009,14038.0,sq mi,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NWIS


In [4]:
# Step 2: Rename columns - strip spaces, replace with underscores
df_station.columns = df_station.columns.str.strip().str.replace(" ", "_")

# Step 3: Drop columns with >90% missing values
df_station.replace(r'^\s*$', pd.NA, regex=True, inplace=True)
threshold = len(df_station) * 0.10
df_station.dropna(axis=1, thresh=threshold, inplace=True)

# Step 4: Select key columns (only keep ones that survived the drop)
keep_cols = [
    "OrganizationIdentifier",
    "MonitoringLocationIdentifier",
    "MonitoringLocationName",
    "MonitoringLocationTypeName",
    "HUCEightDigitCode",
    "LatitudeMeasure",
    "LongitudeMeasure",
    "DrainageAreaMeasure/MeasureValue",
    "DrainageAreaMeasure/MeasureUnitCode",
    "StateCode",
    "CountyCode",
    "ProviderName"
]
keep_cols = [c for c in keep_cols if c in df_station.columns]
df_station = df_station[keep_cols]

# Step 5: Fix data types
df_station["LatitudeMeasure"] = pd.to_numeric(df_station["LatitudeMeasure"], errors="coerce")
df_station["LongitudeMeasure"] = pd.to_numeric(df_station["LongitudeMeasure"], errors="coerce")
if "DrainageAreaMeasure/MeasureValue" in df_station.columns:
    df_station["DrainageAreaMeasure/MeasureValue"] = pd.to_numeric(
        df_station["DrainageAreaMeasure/MeasureValue"], errors="coerce"
    )

print(f"Shape: {df_station.shape}")
print(f"Columns: {df_station.columns.tolist()}")
df_station.dtypes

Shape: (621, 10)
Columns: ['OrganizationIdentifier', 'MonitoringLocationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName', 'HUCEightDigitCode', 'LatitudeMeasure', 'LongitudeMeasure', 'StateCode', 'CountyCode', 'ProviderName']


OrganizationIdentifier              str
MonitoringLocationIdentifier        str
MonitoringLocationName              str
MonitoringLocationTypeName          str
HUCEightDigitCode                 int64
LatitudeMeasure                 float64
LongitudeMeasure                float64
StateCode                         int64
CountyCode                        int64
ProviderName                        str
dtype: object

In [5]:
# Step 6: Save Cleaned Data
import os
out_path = '../../../../data/tabular/water-quality/clean'
os.makedirs(out_path, exist_ok=True)
df_station.to_csv(f'{out_path}/epa-stations-clean.csv', index=False)
print(f"Saved {len(df_station):,} rows → {out_path}/epa-stations-clean.csv")

Saved 621 rows → ../../../../data/tabular/water-quality/clean/epa-stations-clean.csv
